In [5]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [7]:
df = pd.read_csv("qoute_dataset.csv")

In [7]:
df.head()



,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [8]:
df["quote"][0]

'“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”'

In [9]:
df.shape

(3038, 2)

In [13]:
quotes = df["quote"]
quotes.head()


,quote
0,“The world as we have created it is a process ...
1,"“It is our choices, Harry, that show what we t..."
2,“There are only two ways to live your life. On...
3,"“The person, be it gentleman or lady, who has ..."
4,"“Imperfection is beauty, madness is genius and..."


In [14]:
## convert into lowercase
quotes = quotes.str.lower()

In [15]:
## remove puchuation
import string

In [16]:
translator = str.maketrans('','',string.punctuation)

In [17]:
quotes = quotes.apply(lambda x: x.translate(translator))

In [15]:
quotes.head()

,quote
0,“the world as we have created it is a process ...
1,“it is our choices harry that show what we tru...
2,“there are only two ways to live your life one...
3,“the person be it gentleman or lady who has no...
4,“imperfection is beauty madness is genius and ...


In [8]:
##tokenization
from tensorflow.keras.preprocessing.text import Tokenizer

In [18]:
vocab_size = 8978
toknizer = Tokenizer(num_words=vocab_size)
toknizer .fit_on_texts(quotes)

In [19]:
word_index = toknizer.word_index
print(len(word_index))
list (word_index.items())[:10]

8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [20]:
sequence = toknizer.texts_to_sequences(quotes)


In [21]:
for i in range(3):
    print(quotes[i])

“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”
“it is our choices harry that show what we truly are far more than our abilities”
“there are only two ways to live your life one is as though nothing is a miracle the other is as though everything is a miracle”


In [22]:
for i in range(3):
    print(sequence[i])

[713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461]
[947, 7, 70, 871, 373, 9, 433, 21, 19, 465, 14, 294, 52, 54, 70, 3676]
[1337, 14, 53, 201, 714, 3, 81, 15, 36, 37, 7, 29, 329, 93, 7, 5, 1157, 1, 101, 7, 29, 329, 126, 7, 5, 3677]


In [23]:
X = []
y = []

for seq in sequence:
    for i in range(1,len(seq)):
        input_seq = seq[:i]
        output_seq = seq[i]
        X.append(input_seq)
        y.append(output_seq)

In [23]:
len(X)

85270

In [24]:
len(y)

85270

In [24]:
## 745 is max sentence length
max_len = max(len(x) for x in X)
print(max_len)

745


In [25]:
##padding
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_padded = pad_sequences(X, maxlen=max_len, padding='pre')


In [27]:
X_padded[0]

array([  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   

In [28]:
X[0]

[713]

In [29]:
y = np.array(y)

In [30]:
X_padded.shape

(85270, 745)

In [31]:
y.shape

(85270,)

In [26]:
## onehot encoding on y
from tensorflow.keras.utils import to_categorical
y_one_hot = to_categorical(y,num_classes=vocab_size)

In [33]:
y.shape

(85270,)

In [34]:
y_one_hot.shape

(85270, 8978)

In [27]:
##embedding
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,SimpleRNN,LSTM, Dense

In [28]:
embedding_dim = 50
rnn_units = 128


In [37]:

rnn_model = Sequential()

rnn_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len)
)
rnn_model.add(SimpleRNN(units=rnn_units))
rnn_model.add(Dense(units=vocab_size, activation='softmax'))

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [38]:

rnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [39]:

rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [29]:
lstm_model = Sequential()
lstm_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len)
)
lstm_model.add(LSTM(units=rnn_units))
lstm_model.add(Dense(units=vocab_size, activation='softmax'))



/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [30]:
lstm_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

lstm_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [31]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [32]:
import os
from tensorflow.keras.callbacks import ModelCheckpoint

save_dir = "/content/drive/MyDrive/LSTM_Checkpoints"
os.makedirs(save_dir, exist_ok=True)

checkpoint = ModelCheckpoint(
    filepath=os.path.join(save_dir, "epoch_{epoch:02d}.keras"),
    save_freq="epoch",
    save_weights_only=False
)

In [44]:
history_lstm = lstm_model.fit(
    X_padded,
    y_one_hot,
    epochs=100,
    batch_size=128,
    validation_split=0.1,
    callbacks=[checkpoint]
)

Epoch 1/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 53ms/step - accuracy: 0.0389 - loss: 6.7464 - val_accuracy: 0.0460 - val_loss: 6.6793
Epoch 2/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 31s 51ms/step - accuracy: 0.0587 - loss: 6.3203 - val_accuracy: 0.0693 - val_loss: 6.5381
Epoch 3/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 54ms/step - accuracy: 0.0839 - loss: 6.0392 - val_accuracy: 0.0909 - val_loss: 6.4439
Epoch 4/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 54ms/step - accuracy: 0.0990 - loss: 5.8251 - val_accuracy: 0.0957 - val_loss: 6.4179
Epoch 5/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 53ms/step - accuracy: 0.1093 - loss: 5.6444 - val_accuracy: 0.1023 - val_loss: 6.4020
Epoch 6/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 41s 54ms/step - accuracy: 0.1203 - loss: 5.4714 - val_accuracy: 0.1050 - val_loss: 6.4134
Epoch 7/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 54ms/step - accuracy: 0.1285 - loss: 5.3162 - val_accuracy: 0.1091 - val_loss: 6.4383
Epoch 8/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 53ms/step - accuracy: 0.1357 - loss: 5

In [33]:
lstm_model.save("lstm_model_epoch50.h5")

In [34]:
index_to_word = {}
for word, index in word_index.items():
  index_to_word[index] = word

In [35]:
from re import VERBOSE
from tensorflow.keras.preprocessing.sequence import pad_sequences

def predictor(model,toknizer,text,max_len):
  text = text.lower()
  seq = toknizer.texts_to_sequences([text])[0]
  seq = pad_sequences([seq], maxlen =max_len,padding='pre')

  pred = model.predict(seq,verbose= 0)
  pred_index = np.argmax(pred)
  return index_to_word[pred_index]


In [36]:
seed_text = "what are you "
next_word = predictor (lstm_model,toknizer,seed_text,max_len)
print(next_word)

limits”


In [37]:
def generate_text(model,toknizer,seed_text,max_len,n_words):
  for _ in range(n_words):
    next_word = predictor(model,toknizer,seed_text,max_len)
    if next_word == "":
      break
    seed_text += " " + next_word
  return seed_text


In [38]:
seed = "are you a "
generate_text = generate_text(lstm_model,toknizer,seed,max_len,10)
print(generate_text)

are you a  sings gathers gathers inconstant absurdity absurdity greeni daughter daughter confront


In [39]:

import pickle
with open("toknizer.pkl", "wb") as f:
  pickle.dump(toknizer, f)


In [40]:
with open("max_len.pkl", "wb") as f:
  pickle.dump(max_len, f)